# Benchmark model visualisation

Instantiates each model family with several modality configs, prints
architecture summaries, parameter counts per submodule, and forward-pass
tensor shapes. Designed for deep-diving into how each model handles its
inputs.

See `docs/benchmark_models.md` for the canonical architectural reference.


In [ ]:
# Setup
import os, sys
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.getcwd() != REPO_ROOT:
    os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'

import torch
from planner.risk.benchmark_dataset import ModalityConfig
from planner.risk.models import make_model, available_models

print('available models:', available_models())
print('torch:', torch.__version__)


## 1. Parameter counts across model × modality matrix

Quick sanity table — which model is biggest in which configuration?


In [ ]:
def param_count(model):
    return sum(p.numel() for p in model.parameters())

configs = [
    ('state',                 ModalityConfig(state=True), 1),
    ('state',                 ModalityConfig(state=True), 8),
    ('state+goal',            ModalityConfig(state=True, goal=True), 8),
    ('state+rgb+depth',       ModalityConfig(state=True, rgb=True, depth=True), 8),
    ('state+dino',            ModalityConfig(state=True, dino=True), 8),
    ('state+fmode+fjoints',   ModalityConfig(state=True, failure_mode=True, failure_joints=True), 8),
    ('all (kitchen sink)',    ModalityConfig(state=True, goal=True, rgb=True, depth=True,
                                              failure_mode=True, failure_joints=True), 8),
]

print(f'{"modality combo":35s} {"T":>2s} | ', end='')
for m in ['mlp', 'convdec', 'unet', 'transformer']:
    print(f'{m:>12s}', end='')
print()
print('-' * 90)

for name, mc, T in configs:
    print(f'{name:35s} {T:>2d} | ', end='')
    for m in ['mlp', 'convdec', 'unet', 'transformer']:
        try:
            model = make_model(m, modalities=mc, grid_hw=(240, 320), T=T)
            n = param_count(model) / 1e6
            print(f'{n:>11.2f}M', end='')
        except Exception as e:
            err = type(e).__name__
            print(f'{err:>12s}', end='')
    print()


## 2. Submodule parameter breakdown — drill into one model

Pick any (model, modalities, T) combo and see which submodule eats how many
parameters.


In [ ]:
def submodule_breakdown(model, prefix='', depth=2):
    '''Recursive param-count walker. Prints top-level submodules with their counts.'''
    rows = []
    total = sum(p.numel() for p in model.parameters())
    for name, child in model.named_children():
        n = sum(p.numel() for p in child.parameters())
        rows.append((name, n, n / max(total, 1) * 100))
    rows.sort(key=lambda r: -r[1])
    print(f'{"submodule":30s} {"params":>12s}  {"% of total":>10s}')
    print('-' * 60)
    for name, n, pct in rows:
        bar = '█' * int(pct / 2)
        print(f'{name:30s} {n:>12,}  {pct:>9.1f}%  {bar}')
    print(f'{"TOTAL":30s} {total:>12,}')

# Example: UNet with full inputs
mc = ModalityConfig(state=True, rgb=True, depth=True, failure_mode=True, failure_joints=True)
m = make_model('unet', modalities=mc, grid_hw=(240, 320), T=8)
print('UNet state+rgb+depth+failure_mode+failure_joints:')
submodule_breakdown(m)


In [ ]:
# Transformer same view
mc = ModalityConfig(state=True, rgb=True, depth=True, failure_mode=True, failure_joints=True)
m = make_model('transformer', modalities=mc, grid_hw=(240, 320), T=8)
print('Transformer state+rgb+depth+failure_mode+failure_joints:')
submodule_breakdown(m)


In [ ]:
# Drill into the transformer encoder itself
print(m.encoder)


## 3. Forward-pass shape trace

Use forward hooks to log the shape of every submodule's output during one
forward pass. Helpful for understanding the data-flow.


In [ ]:
class ShapeTracer:
    '''Registers forward hooks; logs (module_name, output_shape) for the first
    forward of each module. Call .register() before forward(), .summary() after.'''
    def __init__(self, model):
        self.records = []
        self.handles = []
        self._seen = set()
        for name, module in model.named_modules():
            if module is model or len(list(module.children())) > 0:
                # Only leaf modules; or include containers too if you prefer
                continue
            h = module.register_forward_hook(self._hook(name))
            self.handles.append(h)

    def _hook(self, name):
        def fn(module, inputs, output):
            if name in self._seen:
                return
            self._seen.add(name)
            if isinstance(output, torch.Tensor):
                shape = tuple(output.shape)
            elif isinstance(output, (tuple, list)) and output:
                shape = tuple(output[0].shape) if torch.is_tensor(output[0]) else '...'
            else:
                shape = '...'
            self.records.append((name, type(module).__name__, shape))
        return fn

    def close(self):
        for h in self.handles:
            h.remove()

    def summary(self, max_rows=40):
        print(f'{"module":50s} {"type":18s} output_shape')
        print('-' * 95)
        for name, kind, shape in self.records[:max_rows]:
            print(f'{name[:50]:50s} {kind[:18]:18s} {shape}')
        if len(self.records) > max_rows:
            print(f'... ({len(self.records) - max_rows} more rows)')

def build_dummy_batch(mc, T=8, B=2):
    out = {}
    if mc.state: out['state_window'] = torch.randn(B, T, 18)
    if mc.goal: out['goal'] = torch.randn(B, 3, 11)
    if mc.rgb: out['rgb_window'] = torch.rand(B, T, 3, 240, 320)
    if mc.depth: out['depth_window'] = torch.rand(B, T, 1, 240, 320)
    if mc.dino: out['dino_window'] = torch.randn(B, T, 384)
    if mc.failure_mode: out['failure_mode'] = torch.eye(5)[[0, 1]]
    if mc.failure_joints: out['failure_joints'] = torch.zeros(B, 7).bernoulli_(0.3)
    return out

# Trace UNet late_fusion
mc = ModalityConfig(state=True, rgb=True, depth=True)
m = make_model('unet', modalities=mc, grid_hw=(240, 320), T=8)
m.temporal_mode = 'late_fusion'
m_lf = type(m)(modalities=mc, grid_hw=(240, 320), T=8, temporal_mode='late_fusion', pretrained=False)
tracer = ShapeTracer(m_lf)
with torch.no_grad():
    _ = m_lf(build_dummy_batch(mc, T=8))
tracer.summary(max_rows=30)
tracer.close()


In [ ]:
# Trace Transformer
mc = ModalityConfig(state=True, rgb=True, depth=True)
m = make_model('transformer', modalities=mc, grid_hw=(240, 320), T=8)
tracer = ShapeTracer(m)
with torch.no_grad():
    _ = m(build_dummy_batch(mc, T=8))
tracer.summary(max_rows=30)
tracer.close()


## 4. Visualise a trained prediction

Load any checkpoint from `runs/bench/` and render its predicted heatmap vs
the ground truth on a few val trials. Useful for spot-checking what each
model actually outputs.


In [ ]:
from pathlib import Path
import h5py, hdf5plugin  # noqa: F401

# List available checkpoints
ckpt_root = Path('runs/bench')
ckpts = sorted([p for p in ckpt_root.glob('*/best.pt')
                if not p.parent.name.startswith('_')])
print(f'{len(ckpts)} checkpoints found. Latest 5:')
for p in ckpts[-5:]:
    print(f'  {p.parent.name}')


In [ ]:
# Pick one and render its prediction on 4 val trials.
import numpy as np
import matplotlib.pyplot as plt
from planner.risk.benchmark_dataset import (
    BenchmarkDataset, ModalityConfig, TargetConfig, demo_stratified_split,
)

ckpt_path = ckpts[-1]   # latest checkpoint
print(f'using {ckpt_path}')
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
mod_cfg = ModalityConfig(**ckpt['modalities'])
grid_hw = tuple(ckpt['grid_hw'])
cargs = ckpt['args']

ds = BenchmarkDataset(
    cargs.get('v2_root', '/home/aaron/scratch/v2_ssd'),
    modalities=mod_cfg,
    target_cfg=TargetConfig(log1p=True),
    splits=tuple(cargs.get('splits', ['libero_spatial'])),
    use_window=(cargs.get('T', 8) == 8),
)
_, val_idx = demo_stratified_split(ds, val_frac=0.10, seed=cargs.get('seed', 0))

extra = {}
if cargs['model'] == 'unet':
    extra['temporal_mode'] = cargs.get('unet_temporal', 'mean')
model = make_model(cargs['model'], modalities=mod_cfg, grid_hw=grid_hw,
                   T=cargs.get('T', 8), **extra)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'loaded {cargs["model"]} ({sum(p.numel() for p in model.parameters())/1e6:.2f}M params)')

# Render 4 val trials
rng = np.random.default_rng(0)
sel = rng.choice(val_idx, size=4, replace=False)
fig, axes = plt.subplots(4, 3, figsize=(12, 14))
for r, i in enumerate(sel):
    sample = ds[int(i)]
    batch = {k: torch.from_numpy(v).unsqueeze(0) if isinstance(v, np.ndarray)
             else v for k, v in sample.items()}
    with torch.no_grad():
        out = model(batch)
    pred = out['pred'][0].numpy()
    tgt = sample['target_log1p']
    err = np.abs(pred - tgt)
    vmax = max(pred.max(), tgt.max())
    for c, (img, title) in enumerate([(tgt, 'target log1p'),
                                       (pred, 'pred log1p'),
                                       (err, '|err|')]):
        ax = axes[r, c]
        ax.imshow(img, cmap='magma',
                  vmin=0, vmax=vmax if c < 2 else err.max() + 1e-6)
        ax.set_title(title if r == 0 else '')
        ax.axis('off')
    axes[r, 0].set_ylabel(f"trial {sample.get('trial_id', i)}\nmass={float(sample['target_mass']):.1f}",
                          rotation=0, ha='right', va='center', fontsize=8)
fig.suptitle(f"{ckpt_path.parent.name}", fontsize=10)
plt.tight_layout()
plt.show()


## 5. Compare two model predictions on the same trial

Load two checkpoints and overlay their predictions side-by-side. Useful for
seeing where, e.g., state-only and vision diverge.


In [ ]:
# Pick two specific checkpoints to compare; edit these.
ckpt_a_name = 'convdec__state__T1__seed0__20260526-173653'  # state-only leader
ckpt_b_name = next((p.parent.name for p in ckpts
                    if 'late_fusion' in cargs.get('unet_temporal', '')
                    or 'unet' in p.parent.name), None)
print('A:', ckpt_a_name)
print('B:', ckpt_b_name)

# (Full comparison plot left as an exercise — extend Section 4 to load TWO models
#  and render in a 4×4 grid: (target, pred_A, pred_B, |diff_A_B|) per trial.)
